# Code for the satisfaction final score

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sentence_transformers import SentenceTransformer, util

/users/eleves-a/2022/adrien.bindel/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/users/eleves-a/2022/adrien.bindel/.local/lib/python3.9/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-02-09 15:59:25.700919: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-09 15:59:28.233321: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To e

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", cache_folder='../embedding_models').to(device)

## Study 1

In [3]:
df1 = pd.read_excel('../data/pred_attributes_and_sentiment/study_1_results.xlsx')

In [4]:
df_X = model.encode(
    df1["finalReview"].fillna("").tolist(), 
    # convert_to_tensor=True
    convert_to_tensor=False
)
df_y = df1["Satisfaction_final"]

In [5]:
df1.columns

Index(['ID', 'finalReview', 'Satisfaction_final', 'pred_Satisfaction_final',
       'cleaning_service_quality', 'order_packaging',
       'communication_and_responsiveness', 'Driver_professionalism',
       'Service_speed', 'cleaning_service_quality_sentiment',
       'order_packaging_sentiment',
       'communication_and_responsiveness_sentiment',
       'Driver_professionalism_sentiment', 'Service_speed_sentiment',
       'Overall_review_sentiment', 'Emotional_intensity_LLM', 'lang'],
      dtype='object')

In [6]:
X = df_X
y = df_y.values

clf_1 = Ridge(random_state=42, alpha=1, solver="auto")

clf_1.fit(X, y)

y_pred = clf_1.predict(X)
y_pred_rounded = np.round(y_pred * 2) / 2

mae = mean_absolute_error(y,y_pred)
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print(f"mae: {mae}")
print(f"mae with rounded predictions: {mae_rounded}")

mae: 0.8033246697140692
mae with rounded predictions: 0.7922751729438893


In [7]:
columns = list(df1.columns)

# columns = columns[1:]

columns = columns[:3] +  ["pred_Satisfaction_final"] + columns[3:]

df1["pred_Satisfaction_final"] = y_pred_rounded

df1 = df1.loc[:,columns]

df1.head()

,ID,finalReview,Satisfaction_final,pred_Satisfaction_final,pred_Satisfaction_final,cleaning_service_quality,order_packaging,communication_and_responsiveness,Driver_professionalism,Service_speed,cleaning_service_quality_sentiment,order_packaging_sentiment,communication_and_responsiveness_sentiment,Driver_professionalism_sentiment,Service_speed_sentiment,Overall_review_sentiment,Emotional_intensity_LLM,lang
0,1,My order was to dryclean! All suits came back ...,1.0,3.0,3.0,1,1,1,0,0,4.42,4.19,4.45,NaN,NaN,NaN,NaN,en
1,2,poor experience. jacket not cleaned properly.,2.5,2.5,2.5,1,1,1,0,0,3.98,3.52,4.32,NaN,NaN,NaN,NaN,en
2,3,The clean laundry came in a bag that had a sme...,4.5,4.5,4.5,1,1,0,0,0,3.90,4.38,NaN,NaN,NaN,NaN,NaN,en
3,4,not happy with the service. i received multipl...,3.0,3.5,3.5,1,1,1,0,1,4.56,4.28,4.69,NaN,5.41,NaN,NaN,en
4,5,its a very expensive service.,3.5,3.0,3.0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,en


In [8]:
df1.to_excel('../data/pred_attributes_and_sentiment/study_1_results.xlsx', index=False)

In [9]:
import pickle

# Save the model
with open('../final_pipeline/models/satisfaction_final/ridge_model_study_1.pkl', 'wb') as f:
    pickle.dump(clf_1, f)

# To load it back later:
# with open('ridge_model.pkl', 'rb') as f:
#     loaded_model = pickle.load(f)

## Study 4

In [10]:
df4 = pd.read_csv('../data/cleaned_data/cleaned_Study_4_reviews.csv')

In [11]:
df4.columns

Index(['ID', 'text', 'Satisfaction_RA2', 'Quality_and_taste_of_food',
       'Cleanliness', 'Friendliness_of_staff', 'Value', 'Speed_of_service',
       'Quality_and_taste_of_food_sentiment', 'Cleanliness_sentiment',
       'Friendliness_of_staff_sentiment', 'Value_sentiment',
       'Speed_of_service_sentiment', 'Emotional_intensity_LLM', 'lang'],
      dtype='object')

In [12]:
df_X = model.encode(
    df4["text"].fillna("").tolist(), 
    # convert_to_tensor=True
    convert_to_tensor=False
)
df_y = df4["Satisfaction_RA2"]

In [13]:
X = df_X
y = df_y.values

clf = Ridge(random_state=42, alpha=1, solver="auto")

clf.fit(X, y)

y_pred = clf.predict(X)
y_pred_rounded = np.round(y_pred * 2) / 2

mae = mean_absolute_error(y,y_pred)
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print(f"mae: {mae}")
print(f"mae with rounded predictions: {mae_rounded}")

mae: 0.6390470266342163
mae with rounded predictions: 0.6364365816116333


## Study 5

In [14]:
df5 = pd.read_csv('../data/cleaned_data/cleaned_Study_5_reviews.csv')

In [15]:
df_X = model.encode(
    df5["Review"].fillna("").tolist(), 
    # convert_to_tensor=True
    convert_to_tensor=False
)
df_y = df5["Satisfaction_final"]

In [16]:
X = df_X
y = df_y.values

clf = Ridge(random_state=42, alpha=1, solver="auto")

clf.fit(X, y)

y_pred = clf.predict(X)
y_pred_rounded = np.round(y_pred * 2) / 2

mae = mean_absolute_error(y,y_pred)
mae_rounded = mean_absolute_error(y, y_pred_rounded)

print(f"mae: {mae}")
print(f"mae with rounded predictions: {mae_rounded}")

mae: 0.5900684595108032
mae with rounded predictions: 0.5641562342643738
